# Wetteranalyse: Berlin

Dieses Notebook untersucht historische Tageswerte für Berlin: mittlere, minimale und maximale Temperatur sowie Niederschlag. Es betrachtet automatisch die letzten 14 Tage, für die die Wetterarchiv-Daten voraussichtlich vollständig vorliegen.

Die Koordinaten liegen im Berliner Stadtzentrum. Die Daten stammen von der frei zugänglichen [Open-Meteo-Archiv-API](https://open-meteo.com/). Zum Abruf wird eine Internetverbindung benötigt; ein API-Schlüssel ist nicht erforderlich.

## 1. Zeitraum festlegen

Das Wetterarchiv kann aktuelle Tage mit Verzögerung bereitstellen. Deshalb endet der Zeitraum sieben Tage vor heute und umfasst dann 14 Tage. Du kannst die Werte unten anpassen, falls du einen anderen Zeitraum untersuchen möchtest.

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

heute = datetime.now(ZoneInfo("Europe/Berlin")).date()
enddatum = heute - timedelta(days=7)
startdatum = enddatum - timedelta(days=13)

print(f"Untersuchungszeitraum: {startdatum:%d.%m.%Y} bis {enddatum:%d.%m.%Y}")

## 2. Wetterdaten abrufen

Die API liefert Tageswerte als JSON. Mit `urllib` und `json` verwenden wir dafür nur Python-Bibliotheken, die bereits mit Python kommen.

In [ ]:
import json
from urllib.parse import urlencode
from urllib.request import urlopen

parameter = {
    "latitude": 52.52,
    "longitude": 13.405,
    "start_date": startdatum.isoformat(),
    "end_date": enddatum.isoformat(),
    "daily": "temperature_2m_mean,temperature_2m_min,temperature_2m_max,precipitation_sum",
    "timezone": "Europe/Berlin",
}
url = "https://archive-api.open-meteo.com/v1/archive?" + urlencode(parameter)

with urlopen(url, timeout=30) as antwort:
    wetter = json.load(antwort)

tageswerte = wetter["daily"]
print(f"{len(tageswerte['time'])} Tageswerte geladen.")

## 3. Tageswerte als Tabelle

Temperaturen werden in Grad Celsius und Niederschlag in Millimetern angegeben. `None` bedeutet, dass für den betreffenden Wert keine Angabe vorliegt.

In [ ]:
spalten = [
    ("temperature_2m_min", "Min °C"),
    ("temperature_2m_mean", "Mittel °C"),
    ("temperature_2m_max", "Max °C"),
    ("precipitation_sum", "Regen mm"),
]
kopf = f"{'Datum':<12}" + "".join(f"{name:>12}" for _, name in spalten)
print(kopf)
print("-" * len(kopf))

for index, datum in enumerate(tageswerte["time"]):
    zeile = f"{datum:<12}"
    for schluessel, _ in spalten:
        wert = tageswerte[schluessel][index]
        anzeige = f"{wert:.1f}" if wert is not None else "k.A."
        zeile += f"{anzeige:>12}"
    print(zeile)

## 4. Zusammenfassung berechnen

Wir berechnen den Durchschnitt der Tagesmitteltemperaturen, die höchste Tageshöchsttemperatur, die niedrigste Tagestiefsttemperatur, die Niederschlagssumme und die Anzahl der Regentage. Fehlende Werte werden übersprungen.

In [ ]:
mittelwerte = [wert for wert in tageswerte["temperature_2m_mean"] if wert is not None]
minima = [wert for wert in tageswerte["temperature_2m_min"] if wert is not None]
maxima = [wert for wert in tageswerte["temperature_2m_max"] if wert is not None]
niederschlaege = [wert for wert in tageswerte["precipitation_sum"] if wert is not None]

if mittelwerte:
    print(f"Durchschnittliche Tagesmitteltemperatur: {sum(mittelwerte) / len(mittelwerte):.1f} °C")
if maxima:
    print(f"Höchste Tageshöchsttemperatur: {max(maxima):.1f} °C")
if minima:
    print(f"Niedrigste Tagestiefsttemperatur: {min(minima):.1f} °C")
if niederschlaege:
    regentage = sum(1 for wert in niederschlaege if wert > 0)
    print(f"Niederschlag insgesamt: {sum(niederschlaege):.1f} mm")
    print(f"Tage mit Niederschlag: {regentage} von {len(niederschlaege)}")

## Einordnung

Die Kennzahlen beschreiben nur den ausgewählten Zeitraum und den verwendeten Standort im Berliner Zentrum. Sie sind keine Wettervorhersage. Niederschlagssummen beziehen sich auf den jeweiligen ganzen Tag.

### Übung

Ändere den Zeitraum in der ersten Codezelle und führe anschließend alle Zellen erneut aus. Vergleiche, wie sich Temperatur und Niederschlag verändern.